# Satellite / Multi-Spectral Walkthrough

Phase 4 builds on the Phase 1 multispectral pipeline (16-bit I/O, percentile
normalization, NDVI/NDWI indices, channel attention) with:

- **Band-group stems** (SatMAE strategy 3) — per-spectral-group conv stems.
- **Foundation backbones** SatMAE & Prithvi, registered like any other backbone.
- **Change detection** (Siamese U-Net) and **temporal pooling** over clips.
- A **TorchGeo** adapter for EuroSAT / BigEarthNet.

The from-scratch pieces run offline on CPU; foundation weights and torchgeo
datasets need network.

In [ ]:
import torch
import matplotlib.pyplot as plt
torch.manual_seed(0)

## 1. Band-group stems

`band_groups` in the backbone config routes each spectral group through its
own stem, fused to 3 channels so a standard backbone body consumes it.

In [ ]:
from image_analytics.backbones.registry import build_backbone
from image_analytics.core.config import BackboneConfig

bb = build_backbone(BackboneConfig(name='resnet18', pretrained=False, in_channels=10,
    kwargs={'stem_band_groups': [[0,1,2],[3,4,5,6],[7,8,9]]}))
out = bb(torch.randn(2, 10, 64, 64))
print('10-band -> grouped stems -> resnet18 pooled:', tuple(out.shape))

## 2. Foundation backbones (SatMAE, Prithvi)

Both register in `BACKBONES`. SatMAE patch-embeds per band group with spectral
positional encodings; Prithvi is a temporal ViT taking `(B, C, T, H, W)`.
Shown random-init (offline); set `pretrained=True` to pull hub weights.

In [ ]:
from image_analytics.backbones import BACKBONES
import image_analytics.backbones  # register satellite backbones

satmae = BACKBONES.build('satmae_base', img_size=64, patch_size=16,
                         embed_dim=192, depth=2, num_heads=3)
print('SatMAE (10-band image):', tuple(satmae(torch.randn(2, 10, 64, 64)).shape))

prithvi = BACKBONES.build('prithvi_100m', in_channels=6, img_size=64, patch_size=16,
                          num_frames=3, embed_dim=192, depth=2, num_heads=3)
print('Prithvi (6-band x 3 frames):', tuple(prithvi(torch.randn(2, 6, 3, 64, 64)).shape))

## 3. Change detection (Siamese U-Net)

`synthetic_change` is a shapes scene and a mutated copy; the changed pixels are
the GT mask. The two dates are channel-concatenated, the Siamese encoder shares
weights across them, and per-level feature differences feed a U-Net decoder.

In [ ]:
from image_analytics.data.datasets import build_dataset
from image_analytics.core.config import DataConfig

ds = build_dataset(DataConfig(dataset='synthetic_change', kwargs={'size': 16, 'image_size': 96}), split='train')
img, mask = ds[0]
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(img[:3].permute(1, 2, 0)); ax[0].set_title('t0'); ax[0].axis('off')
ax[1].imshow(img[3:].permute(1, 2, 0)); ax[1].set_title('t1'); ax[1].axis('off')
ax[2].imshow(mask, cmap='gray'); ax[2].set_title('change mask'); ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from image_analytics.core.config import load_config
from image_analytics.segmentation.train import run

config = load_config('../configs/segmentation/change_detection_shapes.yaml', overrides=[
    'training.epochs=5', 'training.device=cpu', 'training.warmup_epochs=0',
    'data.kwargs.size=128', 'model.backbone.pretrained=false', 'output_dir=outputs',
])
metrics = run(config)
print('change-detection mIoU:', round(metrics['val/mIoU'], 3))

## 4. Temporal stacking + pooling

`synthetic_temporal` returns `(C, T, H, W)` clips of a moving shape (label =
motion direction). `temporal_classifier` runs a 2D backbone per frame and pools
features over time (mean / max / attention).

In [ ]:
from image_analytics.core.registry import MODELS
import image_analytics.segmentation.change_detection  # register temporal_classifier

clip_ds = build_dataset(DataConfig(dataset='synthetic_temporal', kwargs={'size': 8, 'image_size': 64, 'num_frames': 4}), split='train')
clip, direction = clip_ds[0]
print('clip (C,T,H,W):', tuple(clip.shape), '| direction:', clip_ds.CLASSES[direction])

backbone = build_backbone(BackboneConfig(name='resnet18', pretrained=False))
model = MODELS.build('temporal_classifier', backbone=backbone, num_classes=4, pool='attention')
print('temporal logits:', tuple(model(clip.unsqueeze(0)).shape))

## 5. Real datasets via TorchGeo

The `torchgeo` dataset key adapts EuroSAT / BigEarthNet (and geo-aware samplers
are in `data/samplers.py`). Templates are checked in — they need the `[geo]`
extra and download on first run:

```bash
python scripts/train.py --config configs/classification/eurosat_ms_scratch.yaml
python scripts/train.py --config configs/classification/eurosat_satmae_linearprobe.yaml
python scripts/train.py --config configs/classification/bigearthnet_multilabel.yaml
```

## Summary

| Component | What |
|---|---|
| Band-group stems | per-spectral-group convs (`backbone.kwargs.band_groups`) |
| `satmae_base` / `prithvi_100m` | foundation backbones in `BACKBONES` |
| `siamese_unet` | bi-temporal change detection (reuses the U-Net decoder) |
| `temporal_classifier` | per-frame backbone + temporal pooling |
| `torchgeo` dataset + `build_geo_sampler` | EuroSAT / BigEarthNet, geo sampling |

Phase 4 completes the satellite story; Phase 6 (serving/MLOps) exports models
from every phase.